We use Monte Carlo simulation to approximate the value of $\pi$

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import random
import math

In [4]:
# Enable interactive widgets
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Create widgets
n_points_slider = widgets.IntSlider(value=10000, min=1000, max=200000, step=1000, description='Points:')
run_button = widgets.Button(description='Run Simulation', button_style='success')
output = widgets.Output()
status = widgets.HTML(value="<b>Status:</b> Ready")

In [5]:
def monte_carlo_pi(n_points):
    inside = 0
    points_inside_x, points_inside_y = [], []
    points_outside_x, points_outside_y = [], []
    
    # For plotting, limit to at most 5000 points each (to avoid clutter)
    max_plot = 5000
    plot_every = max(1, n_points // max_plot)
    
    for i in range(n_points):
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)
        if x*x + y*y <= 1.0:
            inside += 1
            if i % plot_every == 0 and len(points_inside_x) < max_plot:
                points_inside_x.append(x)
                points_inside_y.append(y)
        else:
            if i % plot_every == 0 and len(points_outside_x) < max_plot:
                points_outside_x.append(x)
                points_outside_y.append(y)
    
    pi_approx = 4.0 * inside / n_points
    error = abs(pi_approx - math.pi)
    return pi_approx, error, (points_inside_x, points_inside_y), (points_outside_x, points_outside_y)

def on_run_clicked(b):
    with output:
        clear_output(wait=True)
        n = n_points_slider.value
        status.value = f"<b>Status:</b> Running simulation with {n} points..."
        
        # Run simulation
        pi_approx, error, inside_pts, outside_pts = monte_carlo_pi(n)
        
        # Create plot
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_aspect('equal')
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        
        # Draw unit circle
        theta = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=1, label='Unit circle')
        
        # Plot points
        if inside_pts[0]:
            ax.scatter(inside_pts[0], inside_pts[1], c='red', s=1, alpha=0.6, label=f'Inside ({len(inside_pts[0])} shown)')
        if outside_pts[0]:
            ax.scatter(outside_pts[0], outside_pts[1], c='blue', s=1, alpha=0.6, label=f'Outside ({len(outside_pts[0])} shown)')
        
        ax.legend(loc='upper right', fontsize='small')
        ax.set_title(f'Monte Carlo π ≈ {pi_approx:.6f}\nError = {error:.2e}')
        plt.show()
        
        status.value = f"<b>Status:</b> Done. π ≈ {pi_approx:.8f} (error = {error:.2e})"

# Attach callback
run_button.on_click(on_run_clicked)

# Display GUI
display(widgets.VBox([n_points_slider, run_button, status, output]))